## This is all the import


In [1]:
# verify_setup.py
import torch
import transformers
from torch.cuda import get_device_properties
from accelerate import Accelerator
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"CUDA Version: {torch.version.cuda}")
print(f"CuDNN Version: {torch.backends.cudnn.version()}")
print(f"GPU Count: {torch.cuda.device_count()}")

for i in range(torch.cuda.device_count()):
    props = get_device_properties(i)
    print(f"\nGPU {i}: {props.name}")
    print(f"  Memory: {props.total_memory / 1e9:.1f} GB")
    print(f"  Compute Capability: {props.major}.{props.minor}")
    print(f"  Max Threads per Block: {props}")

# Check VRAM usage
print(f"\nCurrent VRAM Used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"Max VRAM Available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


# set default device
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# torch.set_default_device(device)

PyTorch Version: 2.9.1+cu130
CUDA Available: True
CUDA Version: 13.0
CuDNN Version: 91300
GPU Count: 1

GPU 0: NVIDIA GeForce RTX 3070 Laptop GPU
  Memory: 8.6 GB
  Compute Capability: 8.6
  Max Threads per Block: _CudaDeviceProperties(name='NVIDIA GeForce RTX 3070 Laptop GPU', major=8, minor=6, total_memory=8191MB, multi_processor_count=40, uuid=15c35552-3ae0-1b02-2037-94a90dc53a9b, pci_bus_id=1, pci_device_id=0, pci_domain_id=0, L2_cache_size=4MB)

Current VRAM Used: 0.00 GB
Max VRAM Available: 8.6 GB


In [2]:
from os import remove
import os
import json
from datasets import load_dataset, concatenate_datasets
import random
from pathlib import Path
from transformers import AutoTokenizer

In [1]:
"""
Discrete Diffusion Language Model Architecture
Based on: https://arxiv.org/abs/2211.15029
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from typing import Optional, Tuple, List
from dataclasses import dataclass
from torch.optim import AdamW
from torch.utils.data import DataLoader
from torch.cuda.amp import GradScaler, autocast
from transformers import get_cosine_schedule_with_warmup, AutoTokenizer
from datasets import load_from_disk
from tqdm import tqdm
import os
import json



@dataclass
class DiffusionLLMConfig:
    """Configuration for Diffusion-based LLM"""
    
    # Model architecture
    vocab_size: int = 32000
    hidden_size: int = 576          # ← Reduced for 8GB
    num_hidden_layers: int = 12
    num_attention_heads: int = 9
    num_key_value_heads: int = 3
    intermediate_size: int = 1536
    max_seq_length: int = 2048
    
    # Diffusion parameters
    num_diffusion_steps: int = 100   # ← Key: number of denoising steps
    diffusion_schedule: str = "cosine"  # linear, sqrt, cosine
    noise_pred_type: str = "sample"     # sample, mean, logits
    
    # Training
    timestep_embedding_dim: int = 128
    rms_norm_eps: float = 1e-5
    
    def __post_init__(self):
        assert self.hidden_size % self.num_attention_heads == 0
        assert self.hidden_size % self.num_key_value_heads == 0


class NoiseSchedule:
    """Handles noise schedule for diffusion process."""
    
    def __init__(self, num_steps: int = 50, schedule_type: str = "cosine"):
        self.num_steps = num_steps
        self.schedule_type = schedule_type
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        # print('the device in noise schedule is ', self.device)
        
        # Pre-compute schedules
        if schedule_type == "cosine":
            self.alphas = self._cosine_schedule()
        elif schedule_type == "linear":
            self.alphas = torch.linspace(1.0, 0.0, num_steps)
        elif schedule_type == "sqrt":
            self.alphas = torch.sqrt(torch.linspace(1.0, 0.0, num_steps))
        
        # Precompute cumulative products
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)
        self.alphas_cumprod_prev = torch.cat(
            [torch.ones(1), self.alphas_cumprod[:-1]]
        )
        
        # Precompute betas (noise levels)
        self.betas = 1.0 - self.alphas
        self.sqrt_betas = torch.sqrt(self.betas)
        self.sqrt_alphas = torch.sqrt(self.alphas)
        self.sqrt_alphas_cumprod = torch.sqrt(self.alphas_cumprod)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - self.alphas_cumprod)
    
    def _cosine_schedule(self) -> torch.Tensor:
        """Cosine annealing schedule (better for language)."""
        steps = torch.linspace(0, 1, self.num_steps + 1)
        alphas = torch.cos(((steps + 0.008) / 1.008) * math.pi * 0.5) ** 2
        alphas = alphas / alphas[0]
        return alphas[:-1]
    
    def get_noise_level(self, t: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Get noise level for timestep t.
        
        Args:
            t: timestep indices [batch_size]
        
        Returns:
            sqrt_alpha_t: scaling for signal
            sqrt_one_minus_alpha_t: scaling for noise
        """
        self.sqrt_alphas_cumprod = self.sqrt_alphas_cumprod.to(self.device)
        self.sqrt_one_minus_alphas_cumprod = self.sqrt_one_minus_alphas_cumprod.to(self.device)

        # print(f'the device of t in noise level is {t.device}')
        # print(f'the device of sqrt_alphas_cumprod in noise level is {self.sqrt_alphas_cumprod.device}')
        idx = torch.tensor([-1,1,1])
        sqrt_alpha_t = self.sqrt_alphas_cumprod[t].view(-1, 1, 1)
        sqrt_one_minus_alpha_t = self.sqrt_one_minus_alphas_cumprod[t].view(-1, 1, 1)
        
        return sqrt_alpha_t, sqrt_one_minus_alpha_t


class TimestepEmbedding(nn.Module):
    """Sinusoidal timestep embedding."""
    
    def __init__(self, embedding_dim: int):
        super().__init__()
        self.embedding_dim = embedding_dim
        
        # Positional encoding
        half_dim = embedding_dim // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim) * -emb)
        self.register_buffer("emb", emb)
    
    def forward(self, t: torch.Tensor) -> torch.Tensor:
        """
        Args:
            t: [batch_size] timestep indices
        
        Returns:
            [batch_size, embedding_dim] timestep embeddings
        """
        emb = t[:, None] * self.emb[None, :]  # [batch, half_dim]
        emb = torch.cat([emb.sin(), emb.cos()], dim=-1)
        return emb

class DiffusionAttention(nn.Module):
    """Attention layer with timestep conditioning."""
    
    def __init__(self, config: DiffusionLLMConfig):
        super().__init__()
        self.hidden_size = config.hidden_size
        self.num_attention_heads = config.num_attention_heads
        self.num_key_value_heads = config.num_key_value_heads
        self.head_dim = config.hidden_size // config.num_attention_heads
        
        # Projections
        self.q_proj = nn.Linear(config.hidden_size, config.num_attention_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(config.hidden_size, config.num_key_value_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(config.hidden_size, config.num_key_value_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(config.hidden_size, config.hidden_size, bias=False)
    
    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """Standard causal self-attention."""
        batch_size, seq_len, _ = hidden_states.shape
        
        # Project and reshape
        q = self.q_proj(hidden_states).view(batch_size, seq_len, self.num_attention_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(hidden_states).view(batch_size, seq_len, self.num_key_value_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(hidden_states).view(batch_size, seq_len, self.num_key_value_heads, self.head_dim).transpose(1, 2)
        
        # Repeat for GQA
        num_rep = self.num_attention_heads // self.num_key_value_heads
        k = k.unsqueeze(2).repeat(1, 1, num_rep, 1, 1).reshape(batch_size, self.num_attention_heads, seq_len, self.head_dim)
        v = v.unsqueeze(2).repeat(1, 1, num_rep, 1, 1).reshape(batch_size, self.num_attention_heads, seq_len, self.head_dim)
        
        # # Attention
        # scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        
        # # Causal mask
        # if attention_mask is None:
        #     causal_mask = torch.triu(torch.full((seq_len, seq_len), float('-inf')), diagonal=1)
        #     # print(f'the shape of causal mask is {causal_mask.shape}')
        #     scores = scores + causal_mask.to(scores.device)
        #     # print(f'the shape of scores with casual mask is {scores.shape}')
        # else:
        #     # print('have an attention mask with shape ', attention_mask.shape)
        #     scores = scores + attention_mask
        #     # print(f'the shape of scores with provided attention mask is {scores.shape}')
        
        # attn_weights = F.softmax(scores, dim=-1)
        # # print(f'the shape of attn_weights is {attn_weights.shape} and shape of v is {v.shape}')
        # attn_output = torch.matmul(attn_weights, v)
        # # print('the shape of attn_output before reshape is ', attn_output.shape)
        # # print(f'the sequence length is {seq_len}')
        # Use PyTorch's optimized kernel (Flash Attention / Memory Efficient Attention)
        # This replaces the manual matmul, softmax, and masking logic
        attn_output = torch.nn.functional.scaled_dot_product_attention(
            q, k, v, 
            attn_mask=None, # Set to None for causal to use is_causal=True
            dropout_p=0.0,
            is_causal=False if attention_mask is not None else True
        )
        # Reshape and project
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.hidden_size)

        return self.o_proj(attn_output)

class DiffusionBlock(nn.Module):
    """Transformer block with diffusion conditioning."""
    
    def __init__(self, config: DiffusionLLMConfig):
        super().__init__()
        
        # Pre-norm
        self.input_norm = nn.LayerNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.post_attn_norm = nn.LayerNorm(config.hidden_size, eps=config.rms_norm_eps)
        
        # Attention
        self.attention = DiffusionAttention(config)
        
        # FFN
        self.mlp = nn.Sequential(
            nn.Linear(config.hidden_size, config.intermediate_size),
            nn.SiLU(),
            nn.Linear(config.intermediate_size, config.hidden_size),
        )
        
        # Timestep conditioning (adaptive instance normalization)
        self.time_mlp = nn.Sequential(
            nn.Linear(config.timestep_embedding_dim, config.hidden_size),
            nn.SiLU(),
            nn.Linear(config.hidden_size, config.hidden_size * 2),
        )
    
    def forward(
        self,
        hidden_states: torch.Tensor,
        time_emb: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """
        Args:
            hidden_states: [batch, seq_len, hidden_size]
            time_emb: [batch, timestep_embedding_dim]
            attention_mask: optional causal mask
        
        Returns:
            [batch, seq_len, hidden_size]
        """
        # Get time conditioning (AdaIN style)
        time_cond = self.time_mlp(time_emb)  # [batch, hidden_size * 2]
        time_scale, time_shift = time_cond.chunk(2, dim=-1)  # Each [batch, hidden_size]
        
        # Self-attention with residual
        residual = hidden_states
        hidden_states = self.input_norm(hidden_states)
        hidden_states = self.attention(hidden_states, attention_mask)
        hidden_states = hidden_states + residual
        
        # FFN with residual and time conditioning
        residual = hidden_states
        hidden_states = self.post_attn_norm(hidden_states)
        
        # Apply time conditioning (scale and shift)
        hidden_states = hidden_states * (1.0 + time_scale.unsqueeze(1)) + time_shift.unsqueeze(1)
        
        # FFN
        hidden_states = self.mlp(hidden_states)
        hidden_states = hidden_states + residual
        
        return hidden_states

class DiffusionLanguageModel(nn.Module):
    """Diffusion-based Language Model."""
    
    def __init__(self, config: DiffusionLLMConfig):
        super().__init__()
        self.config = config
        self.tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama_v1.1")
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = " "

        # print(f'the vocab size in model class is {len(self.tokenizer)}')
        # print(f'teh vocab len in cofig is {config.vocab_size}')
        # Embeddings
        self.embed_tokens = nn.Embedding(len(self.tokenizer), config.hidden_size)
        # positional embeddings
        self.position_embeddings = nn.Embedding(
            config.max_seq_length,
            config.hidden_size
        )
        
        # Timestep embedding
        self.timestep_embed = TimestepEmbedding(config.timestep_embedding_dim)
        
        # Transformer blocks
        self.transformer_blocks = nn.ModuleList([
            DiffusionBlock(config) for _ in range(config.num_hidden_layers)
        ])
        
        # Output normalization and projection
        self.final_norm = nn.LayerNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.logits_head = nn.Linear(config.hidden_size, config.vocab_size)
        self.noise_head = nn.Linear(config.hidden_size, config.hidden_size)
        self.x0_head = nn.Linear(config.hidden_size, config.hidden_size)
        
        # Tie embeddings and output
        self.logits_head.weight = self.embed_tokens.weight
        
        # Noise schedule
        self.noise_schedule = NoiseSchedule(config.num_diffusion_steps, config.diffusion_schedule)
        
        # Initialize weights
        self.apply(self._init_weights)

        # choose device
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def _prepare_attention_mask(self, attention_mask, seq_len, device):
        """
        attention_mask: [B, L] (1 = keep, 0 = pad)
        returns: [B, 1, L, L] additive mask
        """
        # Padding mask
        pad_mask = (1.0 - attention_mask[:, None, None, :]) * -1e9

        # Causal mask
        causal_mask = torch.triu(
            torch.full((seq_len, seq_len), -1e9, device=device),
            diagonal=1
        )

        return pad_mask + causal_mask
    
    def forward(
        self,
        x_t: torch.Tensor,
        t: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Forward pass for diffusion model.
        
        Args:
            x_t: [batch, seq_len] noisy token embeddings (already embedded)
            t: [batch] timestep indices
            attention_mask: optional causal mask
        
        Returns:
            logits: [batch, seq_len, vocab_size]
        """
        # print('the shape of x_t in forward is ', x_t.shape)
        batch_size, seq_len, _ = x_t.shape
        
        # Embed timestep
        time_emb = self.timestep_embed(t)  # [batch, timestep_embedding_dim]

        if attention_mask is not None:
            attn_mask = self._prepare_attention_mask(
                attention_mask,seq_len, x_t.device
            )
            # print(f'attention mask prepared with shape {attn_mask.shape}')
        else:
            attn_mask = None

        
        # Pass through transformer
        hidden_states = x_t
        for block in self.transformer_blocks:
            hidden_states = block(hidden_states, time_emb, attn_mask)
        
        # Final output
        hidden_states = self.final_norm(hidden_states)
        # print('the shape of hidden_states before logits is ', hidden_states.shape)
        noise_pred = self.noise_head(hidden_states)
        x0_pred = self.x0_head(hidden_states)

        # print('the shape of noise_pred is ', noise_pred.shape)
        logits = self.logits_head(hidden_states)
        
        return noise_pred,x0_pred

class DiffusionLLMForTraining(nn.Module):
    """Wrapper for training Diffusion LLM."""
    
    def __init__(self, config: DiffusionLLMConfig):
        super().__init__()
        self.config = config
        self.model = DiffusionLanguageModel(config)
        self.noise_schedule = self.model.noise_schedule

        
        # Embeddings (separate to allow token embedding)
        self.embed_tokens = self.model.embed_tokens
    
    def forward(
        self,
        input_ids: torch.Tensor,
        t: Optional[torch.Tensor] = None,
        attention_mask: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Training forward pass.
        
        Args:
            input_ids: [batch, seq_len] clean token IDs
            t: [batch] timesteps (random if None)
        
        Returns:
            logits: [batch, seq_len, vocab_size] model predictions
            loss: scalar loss
        """
        # print(f'The attention mask here train class is {attention_mask}')
        batch_size, seq_len = input_ids.shape
        # print('the shape of input_ids is ', input_ids.shape)
        
        # Sample timesteps if not provided
        if t is None:
            # print(f'the device is {input_ids.device} ')
            t = torch.randint(0, self.config.num_diffusion_steps, (batch_size,)).to(input_ids.device)

        # adding positional embedding
        positions = torch.arange(seq_len, device=input_ids.device)
        positions = positions.unsqueeze(0).expand(batch_size, seq_len)
        # print(f'the shape of positions is {positions.shape}')
        # print(f'the shape of embed tokens is {self.embed_tokens(input_ids).shape}')
        # print(f'the shape of position embeddings is {self.model.position_embeddings(positions).shape}')
        # print(f"Vocab size: {self.embed_tokens.num_embeddings}")
        # print(f"Max positions allowed: {self.model.position_embeddings.num_embeddings}")
        # print(f"Max position index: {positions.max().item()}")
        # print(f"Max input_id: {input_ids.max().item()}")


        # Embed clean tokens
        x_0 = self.embed_tokens(input_ids) + self.model.position_embeddings(positions)  # [batch, seq_len, hidden_size] [2,2048,576] 
        # print('the shape of x_0 is ', x_0.shape)
        # Sample noise
        noise = torch.randn_like(x_0)
        # print('the shape of noise is ', noise.shape)
        
        # Add noise (forward diffusion process)
        sqrt_alpha_t, sqrt_one_minus_alpha_t = self.noise_schedule.get_noise_level(t)
        sqrt_alpha_t = sqrt_alpha_t.view(batch_size, 1, 1).to(x_0.device)
        sqrt_one_minus_alpha_t = sqrt_one_minus_alpha_t.view(batch_size, 1, 1).to(x_0.device)        
        x_t = sqrt_alpha_t * x_0 + sqrt_one_minus_alpha_t * noise # [batch, seq_len, hidden_size] [2,2048,576] 
        
        # Forward through model (predict noise or mean)
        # print('the shape of x_t is ', x_t.shape)
        # print('the shape of t is ', t.shape) # [batch]
        # logits = self.model(x_t, t) # [batch, seq_len, vocab_size] [2,2048,32000]
        # print('the shape of logits is ', logits.shape)
        attention_mask_full = self.model._prepare_attention_mask(
            attention_mask,
            seq_len,
            input_ids.device
        )
        # predicted noise
        pred_noise,pred_x0 = self.model(x_t,t,attention_mask)
        # print('the shape of pred noise is ', pred_noise.shape)
        
        # MSE loss on noise prediction (standard diffusion loss)
        # In practice, we'd use cross-entropy on discrete tokens, but for now simple MSE
        loss_mse = F.mse_loss(pred_noise, noise)

        logits = self.model.logits_head(pred_x0)
        loss_ce = F.cross_entropy(
            logits.view(-1, logits.size(-1)),
            input_ids.view(-1),
            ignore_index=self.model.tokenizer.pad_token_id
        )

        t_norm = t.float() / self.config.num_diffusion_steps
        lambda_t = (1.0 - t_norm).mean() 

        loss = (lambda_t * loss_ce) + ((1.0 - lambda_t) * loss_mse)  
        return pred_noise, loss
    
    @torch.no_grad()
    def generate(
        self,
        prompt: str = '',
        batch_size: int = 1,
        seq_len: int = 256,
        num_inference_steps: int = 50,
        temperature: float = 1.0,
        guidance_scale: float = 1.0,
    ) -> list:
        """
        Generate text using reverse diffusion process.
        
        Args:
            prompt: Optional input text to condition generation (e.g., "The future of AI is")
            batch_size: number of sequences to generate
            seq_len: total sequence length (including prompt if provided)
            num_inference_steps: number of denoising steps (more = better quality)
            temperature: sampling temperature for final decoding
            guidance_scale: strength of prompt conditioning (higher = stronger)
        
        Returns:
            list of generated texts
        
        Examples:
            # Unconditional generation
            >>> texts = generator.generate(batch_size=2, seq_len=256)
            
            # Prompt-conditioned generation
            >>> texts = generator.generate(
            ...     prompt="The future of artificial intelligence is",
            ...     seq_len=256,
            ...     num_inference_steps=50
            ... )
        """
        
        device = self.model.embed_tokens.weight.device
        
        # Handle prompt if provided
        if prompt is not None:
            # Tokenize prompt
            prompt_tokens = self.model.tokenizer(
                prompt,
                return_tensors="pt",
                add_special_tokens=True,
                truncation=True,
                max_length=seq_len - 10  # Leave room for generation
            )
            prompt_ids = prompt_tokens["input_ids"].to(device)
            prompt_len = prompt_ids.shape[1]
            
            # Replicate prompt for batch
            prompt_ids = prompt_ids.repeat(batch_size, 1)  # [batch_size, prompt_len]
            
            # Get prompt embeddings (these will be fixed during generation)
            prompt_embeddings = self.model.embed_tokens(prompt_ids)  # [batch_size, prompt_len, hidden]
            print(f'the shape of prompt embeddings is {prompt_embeddings.shape}')
            # Length to generate (continuation only)
            generation_len = seq_len - prompt_len
            
            if generation_len <= 0:
                raise ValueError(f"Prompt is too long ({prompt_len} tokens). "
                               f"Reduce prompt or increase seq_len (currently {seq_len})")
            
            print(f"Prompt: '{prompt}'")
            print(f"Prompt length: {prompt_len} tokens")
            print(f"Generating: {generation_len} tokens")
        else:
            # Unconditional generation (no prompt)
            prompt_embeddings = None
            prompt_ids = None
            prompt_len = 0
            generation_len = seq_len
        
        # Start from pure noise for the generation part
        if prompt_embeddings is not None:
            # Only generate noise for continuation
            x_t = torch.randn(batch_size, generation_len, self.config.hidden_size).to(device)
            # Concatenate with prompt embeddings
            x_t = torch.cat([prompt_embeddings, x_t], dim=1)  # [batch, seq_len, hidden]
        else:
            # Full sequence is noise
            x_t = torch.randn(batch_size, seq_len, self.config.hidden_size).to(device)
            positions = torch.arange(seq_len, device=device)
            positions = positions.unsqueeze(0).expand(batch_size, seq_len)

            pos_emb = self.model.position_embeddings(positions)
            x_t = x_t + pos_emb
        
        # Reverse diffusion process
        print(f"\nStarting diffusion generation with {num_inference_steps} steps...")
        for step in range(num_inference_steps - 1, -1, -1):
            t = torch.full((batch_size,), step, dtype=torch.long).to(device)
            
            # Predict noise
            noise_pred = self._predict_noise(x_t, t)
            
            # Denoise
            alpha_t = self.model.noise_schedule.alphas[step]
            alpha_t_prev = self.model.noise_schedule.alphas_cumprod_prev[step] if step > 0 else torch.tensor(1.0)
            
            # DDPM reverse step
            pred_x0 = (x_t - (1 - alpha_t) ** 0.5 * noise_pred) / (alpha_t ** 0.5)
            
            if step > 0:
                noise = torch.randn_like(x_t) * temperature
                x_t = (alpha_t_prev ** 0.5) * pred_x0 + ((1 - alpha_t_prev) ** 0.5) * noise
            else:
                x_t = pred_x0
            
            # Keep prompt embeddings fixed (if provided)
            if prompt_embeddings is not None:
                x_t[:, :prompt_len, :] = prompt_embeddings
            
            if (step + 1) % 10 == 0 or step == 0:
                print(f"  Step {num_inference_steps - step}/{num_inference_steps} complete")
        
        # Decode embeddings to token IDs
        x_0 = x_t
        
        print(f"\nDecoding embeddings to tokens...")
        # Nearest neighbor in embedding space
        distances = torch.cdist(
            x_0.reshape(-1, self.config.hidden_size),
            self.model.embed_tokens.weight
        )  # [batch*seq, vocab]
        
        token_ids = torch.argmin(distances, dim=1)
        token_ids = token_ids.reshape(batch_size, seq_len)
        
        # If we had a prompt, verify it's preserved (for debugging)
        if prompt_ids is not None:
            # Replace generated prompt tokens with original (ensure consistency)
            token_ids[:, :prompt_len] = prompt_ids
        
        # Decode to text
        texts = []
        for ids in token_ids:
            text = self.model.tokenizer.decode(ids, skip_special_tokens=True)
            texts.append(text)
        
        return texts
    
    def _predict_noise(self, x_t: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """
        Predict noise from current state.
        
        Args:
            x_t: current noisy embeddings [batch, seq_len, hidden]
            t: timestep [batch]
        
        Returns:
            predicted noise [batch, seq_len, hidden]
        """
        # Forward through model to get noise prediction
        # Note: You may need to adapt this based on your actual model architecture
        # This assumes your model outputs noise directly
        seq_len = x_t.shape[1]

        causal_mask = torch.triu(
            torch.full((seq_len, seq_len), -1e9, device=x_t.device),
            diagonal=1
        )[None, None, :, :]
        output,_ = self.model(x_t, t, attention_mask=causal_mask)  # Adjust based on your architecture

        return output

# training the LLM

In [ ]:
# train_diffusion_llm_8gb.py
"""
Training script for Diffusion-based LLM on 8GB VRAM
Smaller model: 300M parameters with diffusion
"""




# # Import from architecture file
# from diffusion_llm_architecture import DiffusionLLMConfig, DiffusionLLMForTraining


class DiffusionTrainingConfig:
    """Training configuration for 8GB GPU."""
    
    # Model
    vocab_size = 32000
    hidden_size = 768        # Smaller than standard
    num_hidden_layers = 12
    num_attention_heads = 12
    num_key_value_heads = 4
    intermediate_size = 3072
    max_seq_length = 2048
    
    # Diffusion
    num_diffusion_steps = 125
    diffusion_schedule = "cosine"
    timestep_embedding_dim = 128
    
    # Training
    num_epochs = 10
    batch_size = 4
    gradient_accumulation_steps = 4
    learning_rate = 5e-4
    warmup_steps = 200
    max_grad_norm = 1.0
    weight_decay = 0.01
    
    # Optimization
    use_gradient_checkpointing = True
    use_flash_attention = False  # Not critical for diffusion
    mixed_precision = "fp16"
    use_8bit_optimizer = True
    
    # Logging
    logging_steps = 500
    save_steps = 10000
    eval_steps = 5000
    
    # Data
    dataset_path = "./data/tokenized"
    output_dir = "./diffusion_checkpoints"
    
    # Hardware
    device = "cuda"
    num_workers = 2


def setup_model_and_tokenizer(config: DiffusionTrainingConfig):
    """Initialize model and tokenizer."""
    
    # Create config
    model_config = DiffusionLLMConfig(
        vocab_size=config.vocab_size,
        hidden_size=config.hidden_size,
        num_hidden_layers=config.num_hidden_layers,
        num_attention_heads=config.num_attention_heads,
        num_key_value_heads=config.num_key_value_heads,
        intermediate_size=config.intermediate_size,
        max_seq_length=config.max_seq_length,
        num_diffusion_steps=config.num_diffusion_steps,
        diffusion_schedule=config.diffusion_schedule,
        timestep_embedding_dim=config.timestep_embedding_dim,
    )
    
    # Create model
    model = DiffusionLLMForTraining(model_config)
    
    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Model Parameters: {total_params / 1e9:.3f}B")
    
    # Load tokenizer (for reference)
    tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama_v1.1")
    if tokenizer.pad_token is None:
        tokenizer.pad_token = " "
    
    return model, tokenizer, model_config


def collate_fn(batch):
    """Convert list of lists → stacked tensors."""
    return {
        'input_ids': torch.tensor([item['input_ids'] for item in batch], dtype=torch.long),
        'attention_mask': torch.tensor([item['attention_mask'] for item in batch], dtype=torch.long),
        'labels': torch.tensor([item['labels'] for item in batch], dtype=torch.long),
    }


def train_diffusion_lm():
    """Main training loop."""

    
    config = DiffusionTrainingConfig()
    os.makedirs(config.output_dir, exist_ok=True)
    
    print("[1/5] Setting up model and tokenizer...")
    model, tokenizer, model_config = setup_model_and_tokenizer(config)
    
    # Move to GPU
    model = model.to(config.device)

    # NEW: Compile the model (Requires PyTorch 2.0+)
    # This fuses kernels together, making them 20-30% faster
    if hasattr(torch, 'compile'):
        print("Compiling model for speed...")
        model = torch.compile(model)
    
    print("[2/5] Loading datasets...")
    train_dataset = load_from_disk(f"{config.dataset_path}/train")
    eval_dataset = load_from_disk(f"{config.dataset_path}/test")
    
    # Subset for 8GB training
    # train_dataset = train_dataset.select(range(min(45000, len(train_dataset))))
    # eval_dataset = eval_dataset.select(range(min(3000, len(eval_dataset))))
    # print(f"the training dataset is {train_dataset} ")

    print(f"Training samples: {len(train_dataset)}")
    # print(f'the columns are {train_dataset.column_names} \n and the {torch.tensor(train_dataset['input_ids']).shape}')
    print(f"Evaluation samples: {len(eval_dataset)}")
    
    print("[3/5] Creating dataloaders...")
    train_loader = DataLoader(
        train_dataset,
        batch_size=config.batch_size,
        shuffle=True,
        num_workers=8,           # Increase to 8 for your laptop
        pin_memory=True,         # CRITICAL: Allows direct memory access to GPU
        prefetch_factor=2,       # Pre-loads batches in advance
        persistent_workers=True, # Keeps workers alive (faster start of epoch)
        drop_last=True,
        collate_fn=collate_fn,
    )    
    eval_loader = DataLoader(
        eval_dataset,
        batch_size=config.batch_size,
        shuffle=True,
        num_workers=8,           # Increase to 8 for your laptop
        pin_memory=True,         # CRITICAL: Allows direct memory access to GPU
        prefetch_factor=2,       # Pre-loads batches in advance
        persistent_workers=True, # Keeps workers alive (faster start of epoch)
        drop_last=True,
        collate_fn=collate_fn,
    )
    print(f'the train dataset loader is {train_loader}')
    
    print("[4/5] Setting up optimizer and scheduler...")
    
    # Use 8-bit optimizer to save memory
    from bitsandbytes.optim import AdamW8bit
    
    optimizer = AdamW8bit(
        model.parameters(),
        lr=config.learning_rate,
        weight_decay=config.weight_decay,
        betas=(0.9, 0.95),
    )
    
    total_steps = len(train_loader) * config.num_epochs // config.gradient_accumulation_steps
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=config.warmup_steps,
        num_training_steps=total_steps,
    )
    
    # Mixed precision scaler
    scaler = GradScaler()

    # model, optimizer, train_loader, eval_loader, scheduler  = accelerator.prepare(
    #     model, optimizer, train_loader, eval_loader, scheduler
    # )
    
    # Save config
    with open(f"{config.output_dir}/training_config.json", "w") as f:
        json.dump({
            "model_config": model_config.__dict__,
            "training_config": config.__dict__,
        }, f, indent=2, default=str)
    
    print("\n" + "="*70)
    print("DIFFUSION LLM TRAINING STARTED (8GB VRAM)")
    print("="*70)
    # print(f"Model: {total_params / 1e9:.3f}B parameters")
    print(f"Dataset: {len(train_dataset)} training samples")
    print(f"Batch size: {config.batch_size}")
    print(f"Gradient accumulation: {config.gradient_accumulation_steps}")
    print(f"Effective batch size: {config.batch_size * config.gradient_accumulation_steps}")
    print(f"Diffusion steps: {config.num_diffusion_steps}")
    print(f"Total training steps: {total_steps}")
    print("="*70 + "\n")
    
    # Training loop
    for epoch in range(config.num_epochs):
        model.train()
        total_loss = 0
        
        # progress bar
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{config.num_epochs}")
        # for batch in train_loader:
        #     print(f"Preparing batch...{(batch.keys())}, \n the keys are {batch["input_ids"].shape}")

        
        for step, batch in enumerate(pbar):
            # Prepare batch
            # print(f"Preparing batch...{(batch)}, \n the keys are {batch.keys()}")
            # print(f"the inpuit batch for input ids of len {batch["input_ids"].shape} ")
            # print(f"the inpuit batch for attention mask of len {batch["attention_mask"].shape}")
            # print(f"the inpuit batch for labels of len {batch["labels"].shape} ")
            input_ids = batch["input_ids"].to(config.device)
            attention_mask = batch["attention_mask"].to(config.device)
            
            # Forward pass with mixed precision
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                logits, loss = model(input_ids, attention_mask=attention_mask)
                loss = loss / config.gradient_accumulation_steps
            
            # Backward pass
            # scaler.scale(loss).backward()
            loss.backward()
            total_loss += loss.item() * config.gradient_accumulation_steps
            
            # Gradient accumulation step
            if (step + 1) % config.gradient_accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), config.max_grad_norm)
                # scaler.step(optimizer)
                # scaler.update()
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
            
            # Logging
            if (step + 1) % config.logging_steps == 0:
                avg_loss = total_loss / config.logging_steps
                vram_active = torch.cuda.memory_allocated() / 1e9
                vram_reserved = torch.cuda.memory_reserved() / 1e9  # This will be much closer to nvidia-smi
                pbar.set_postfix({
                    "loss": f"{avg_loss:.4f}",
                    "vram_active": f"{vram_active:.2f}GB",
                    "vram_reserved": f"{vram_reserved:.2f}GB",
                    "lr": f"{scheduler.get_last_lr()[0]:.2e}"
                })
                total_loss = 0
            
            # Save checkpoint
            if (step + 1) % config.save_steps == 0:
                checkpoint_dir = f"{config.output_dir}/checkpoint-{epoch}-{step}"
                os.makedirs(checkpoint_dir, exist_ok=True)
                torch.save(model.state_dict(), f"{checkpoint_dir}/model.pt")
                print(f"\nSaved checkpoint to {checkpoint_dir}")
        
        # Evaluation
        print("\nEvaluating...")
        model.eval()
        eval_loss = 0
        
        with torch.inference_mode():
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                for batch in tqdm(eval_loader, desc="Evaluating"):
                    input_ids = batch["input_ids"].to(config.device)
                    attention_mask = batch["attention_mask"].to(config.device)
                    _, loss = model(input_ids, attention_mask=attention_mask)
                    eval_loss += loss.item()
        
        avg_eval_loss = eval_loss / len(eval_loader)
        print(f"Epoch {epoch+1} - Eval Loss: {avg_eval_loss:.4f}\n")
    
    # Final save
    final_dir = f"{config.output_dir}/final"
    os.makedirs(final_dir, exist_ok=True)
    torch.save(model.state_dict(), f"{final_dir}/model.pt")
    print(f"\nTraining complete! Model saved to {final_dir}")


if __name__ == "__main__":
    train_diffusion_lm()

[1/5] Setting up model and tokenizer...
Model Parameters: 0.118B
Compiling model for speed...
[2/5] Loading datasets...
the training dataset is Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 148355
}) 
Training samples: 148355
the columns are ['input_ids', 'attention_mask', 'labels'] 
 and the torch.Size([148355, 2048])
Evaluation samples: 23425
[3/5] Creating dataloaders...
the train dataset loader is <torch.utils.data.dataloader.DataLoader object at 0x76bd5c63b230>
[4/5] Setting up optimizer and scheduler...


/tmp/ipykernel_11238/2068300098.py:178: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()



DIFFUSION LLM TRAINING STARTED (8GB VRAM)
Dataset: 148355 training samples
Batch size: 4
Gradient accumulation: 4
Effective batch size: 16
Diffusion steps: 125
Total training steps: 92720



Epoch 1/10:  27%|██▋       | 10000/37088 [1:16:26<8:55:57,  1.19s/it, loss=3.2223, vram_active=0.77GB, vram_reserved=7.87GB, lr=4.99e-04]


Saved checkpoint to ./diffusion_checkpoints/checkpoint-0-9999


...

Epoch 1/10:  81%|████████  | 30000/37088 [3:59:17<2:12:01,  1.12s/it, loss=2.8681, vram_active=0.77GB, vram_reserved=7.87GB, lr=4.92e-04]


Saved checkpoint to ./diffusion_checkpoints/checkpoint-0-29999


Epoch 1/10: 100%|██████████| 37088/37088 [4:57:10<00:00,  2.08it/s, loss=2.9089, vram_active=0.77GB, vram_reserved=7.87GB, lr=4.88e-04]  



Evaluating...


Evaluating:   0%|          | 0/5856 [00:00<?, ?it/s]/mnt/c/Users/nikil/Documents/Projects/Train_LLM/train_llm/lib/python3.13/site-packages/torch/_inductor/compile_fx.py:312: UserWarning: TensorFloat32 tensor cores for float32 matrix multiplication available but not enabled. Consider setting `torch.set_float32_matmul_precision('high')` for better performance.
  warnings.warn(
W1220 06:35:16.252000 11238 torch/_inductor/utils.py:1613] [0/2_1] Not enough SMs to use max_autotune_gemm mode
Evaluating:  62%|██████▏   | 3647/5856 [3:25:57<5:44:51,  9.37s/it]  

## generating the output from diffusion model

In [ ]:
class DiffusionLLMGenerator:
    """Generate text using Diffusion LLM with optional prompt conditioning."""
    
    def __init__(self, model_path: str, config_dict: dict):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        # Load config
        model_config = DiffusionLLMConfig(**config_dict)
        
        # Load model
        self.model = DiffusionLLMForTraining(model_config).to(self.device)
        self.model.load_state_dict(torch.load(model_path, map_location=self.device))
        self.model.eval()
        
        # Load tokenizer
        self.tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama_v1.1")
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = " "
        
        # Store config for later use
        self.config = model_config
        
    @torch.no_grad()
    def generate(
        self,
        prompt: str = '',
        batch_size: int = 1,
        seq_len: int = 256,
        num_inference_steps: int = 50,
        temperature: float = 1.0,
        guidance_scale: float = 1.0,
    ) -> list:
        """
        Generate text using reverse diffusion process.
        
        Args:
            prompt: Optional input text to condition generation (e.g., "The future of AI is")
            batch_size: number of sequences to generate
            seq_len: total sequence length (including prompt if provided)
            num_inference_steps: number of denoising steps (more = better quality)
            temperature: sampling temperature for final decoding
            guidance_scale: strength of prompt conditioning (higher = stronger)
        
        Returns:
            list of generated texts
        
        Examples:
            # Unconditional generation
            >>> texts = generator.generate(batch_size=2, seq_len=256)
            
            # Prompt-conditioned generation
            >>> texts = generator.generate(
            ...     prompt="The future of artificial intelligence is",
            ...     seq_len=256,
            ...     num_inference_steps=50
            ... )
        """
        
        device = self.model.embed_tokens.weight.device
        
        # Handle prompt if provided
        if prompt is not None:
            # Tokenize prompt
            prompt_tokens = self.tokenizer(
                prompt,
                return_tensors="pt",
                add_special_tokens=True,
                truncation=True,
                max_length=seq_len - 10  # Leave room for generation
            )
            prompt_ids = prompt_tokens["input_ids"].to(device)
            prompt_len = prompt_ids.shape[1]
            
            # Replicate prompt for batch
            prompt_ids = prompt_ids.repeat(batch_size, 1)  # [batch_size, prompt_len]
            
            # Get prompt embeddings (these will be fixed during generation)
            prompt_embeddings = self.model.embed_tokens(prompt_ids)  # [batch_size, prompt_len, hidden]
            print(f'the shape of prompt embeddings is {prompt_embeddings.shape}')
            # Length to generate (continuation only)
            generation_len = seq_len - prompt_len
            
            if generation_len <= 0:
                raise ValueError(f"Prompt is too long ({prompt_len} tokens). "
                               f"Reduce prompt or increase seq_len (currently {seq_len})")
            
            print(f"Prompt: '{prompt}'")
            print(f"Prompt length: {prompt_len} tokens")
            print(f"Generating: {generation_len} tokens")
        else:
            # Unconditional generation (no prompt)
            prompt_embeddings = None
            prompt_ids = None
            prompt_len = 0
            generation_len = seq_len
        
        # Start from pure noise for the generation part
        if prompt_embeddings is not None:
            # Only generate noise for continuation
            x_t = torch.randn(batch_size, generation_len, self.config.hidden_size).to(device)
            # Concatenate with prompt embeddings
            x_t = torch.cat([prompt_embeddings, x_t], dim=1)  # [batch, seq_len, hidden]
        else:
            # Full sequence is noise
            x_t = torch.randn(batch_size, seq_len, self.config.hidden_size).to(device)
        
        # Reverse diffusion process
        print(f"\nStarting diffusion generation with {num_inference_steps} steps...")
        for step in range(num_inference_steps - 1, -1, -1):
            t = torch.full((batch_size,), step, dtype=torch.long).to(device)
            
            # Predict noise
            noise_pred = self._predict_noise(x_t, t)
            
            # Denoise
            alpha_t = self.model.noise_schedule.alphas[step]
            alpha_t_prev = self.model.noise_schedule.alphas_cumprod_prev[step] if step > 0 else torch.tensor(1.0)
            
            # DDPM reverse step
            pred_x0 = (x_t - (1 - alpha_t) ** 0.5 * noise_pred) / (alpha_t ** 0.5)
            
            if step > 0:
                noise = torch.randn_like(x_t) * temperature
                x_t = (alpha_t_prev ** 0.5) * pred_x0 + ((1 - alpha_t_prev) ** 0.5) * noise
            else:
                x_t = pred_x0
            
            # Keep prompt embeddings fixed (if provided)
            if prompt_embeddings is not None:
                x_t[:, :prompt_len, :] = prompt_embeddings
            
            if (step + 1) % 10 == 0 or step == 0:
                print(f"  Step {num_inference_steps - step}/{num_inference_steps} complete")
        
        # Decode embeddings to token IDs
        x_0 = x_t
        
        print(f"\nDecoding embeddings to tokens...")
        # Nearest neighbor in embedding space
        distances = torch.cdist(
            x_0.reshape(-1, self.config.hidden_size),
            self.model.embed_tokens.weight
        )  # [batch*seq, vocab]
        
        token_ids = torch.argmin(distances, dim=1)
        token_ids = token_ids.reshape(batch_size, seq_len)
        
        # If we had a prompt, verify it's preserved (for debugging)
        if prompt_ids is not None:
            # Replace generated prompt tokens with original (ensure consistency)
            token_ids[:, :prompt_len] = prompt_ids
        
        # Decode to text
        texts = []
        for ids in token_ids:
            text = self.tokenizer.decode(ids, skip_special_tokens=True)
            texts.append(text)
        
        return texts
    
    def _predict_noise(self, x_t: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """
        Predict noise from current state.
        
        Args:
            x_t: current noisy embeddings [batch, seq_len, hidden]
            t: timestep [batch]
        
        Returns:
            predicted noise [batch, seq_len, hidden]
        """
        # Forward through model to get noise prediction
        # Note: You may need to adapt this based on your actual model architecture
        # This assumes your model outputs noise directly
        seq_len = x_t.shape[1]

        causal_mask = torch.triu(
            torch.full((seq_len, seq_len), -1e9, device=x_t.device),
            diagonal=1
        )[None, None, :, :]
        output,_ = self.model.model(x_t, t, attention_mask=causal_mask)  # Adjust based on your architecture

        return output


# Usage examples
if __name__ == "__main__":
    import json
    
    # Load configuration
    with open("./diffusion_checkpoints/training_config.json", "r") as f:
        config_data = json.load(f)
    
    # Create generator
    generator = DiffusionLLMGenerator(
        "./diffusion_checkpoints/final/model.pt",
        config_data["model_config"]
    )
    
    print("="*70)
    print("DIFFUSION LLM GENERATION EXAMPLES")
    print("="*70)
    
    # Example 1: Unconditional generation
    print("\n[1] UNCONDITIONAL GENERATION")
    print("-" * 70)
    texts = generator.generate(
        batch_size=2,
        seq_len=128,
        num_inference_steps=50
    )
    for i, text in enumerate(texts):
        print(f"\nSample {i+1}:")
        print(text)
    
    # Example 2: Prompt-conditioned generation (SHORT PROMPT)
    print("\n\n[2] SHORT PROMPT GENERATION")
    print("-" * 70)
    texts = generator.generate(
        prompt="How are you?",
        batch_size=2,
        seq_len=256,
        num_inference_steps=10
    )
    for i, text in enumerate(texts):
        print(f"\nGeneration {i+1}:")
        print(text)
    
    # Example 3: Prompt-conditioned generation (LONGER PROMPT)
    print("\n\n[3] LONGER PROMPT GENERATION")
    print("-" * 70)
    texts = generator.generate(
        prompt="Artificial intelligence has revolutionized many industries. In healthcare,",
        batch_size=1,
        seq_len=512,
        num_inference_steps=10
    )
    print(f"\nGeneration:")
    print(texts[0])
    
    # # Example 4: Different quality levels with prompt
    # print("\n\n[4] QUALITY COMPARISON WITH PROMPT")
    # print("-" * 70)
    # prompt = "Deep learning models are"
    
    # for steps, quality in [(10, "Fast"), (25, "Balanced"), (50, "High"), (100, "Ultra")]:
    #     print(f"\n{quality} quality ({steps} steps):")
    #     texts = generator.generate(
    #         prompt=prompt,
    #         batch_size=1,
    #         seq_len=200,
    #         num_inference_steps=steps
    #     )
    #     print(texts[0])
    
    # # Example 5: Interactive generation
    # print("\n\n[5] INTERACTIVE GENERATION")
    # print("-" * 70)
    # print("Enter your prompts (or 'quit' to exit):\n")
    
    # while True:
    #     user_prompt = input("Prompt: ").strip()
        
    #     if user_prompt.lower() in ['quit', 'exit', 'q']:
    #         break
        
    #     if not user_prompt:
    #         print("Please enter a prompt.")
    #         continue
        
    #     try:
    #         texts = generator.generate(
    #             prompt=user_prompt,
    #             batch_size=1,
    #             seq_len=256,
    #             num_inference_steps=50
    #         )
    #         print(f"\nGenerated:")
    #         print(texts[0])
    #         print()
    #     except Exception as e:
    #         print(f"Error: {e}\n")


DIFFUSION LLM GENERATION EXAMPLES

[1] UNCONDITIONAL GENERATION
----------------------------------------------------------------------
the shape of prompt embeddings is torch.Size([2, 1, 576])
Prompt: ''
Prompt length: 1 tokens
Generating: 127 tokens

Starting diffusion generation with 50 steps...
the shape of hidden_states before logits is  torch.Size([2, 128, 576])
  Step 1/50 complete
the shape of hidden_states before logits is  torch.Size([2, 128, 576])
the shape of hidden_states before logits is  torch.Size([2, 128, 576])
the shape of hidden_states before logits is  torch.Size([2, 128, 576])
the shape of hidden_states before logits is  torch.Size([2, 128, 576])
the shape of hidden_states before logits is  torch.Size([2, 128, 576])
the shape of hidden_states before logits is  torch.Size([2, 128, 576])
the shape of hidden_states before logits is  torch.Size([2, 128, 576])
the shape of hidden_states before logits is  torch.Size([2, 128, 576])
the shape of hidden_states before logits 